# Ground Truth Validation with spectral_select

This notebook demonstrates how to validate clustering results against ground truth annotations using the `spectral_select` library.

**What you'll learn:**
- Load ground truth from annotated PNG images
- Compute validation metrics (ARI, NMI, purity, etc.)
- Generate validation reports
- Visualize validation results

## 1. Setup

Import the validation classes from `spectral_select`:

In [ ]:
from spectral_select import Validator, Visualizer, load_ground_truth_from_png
import numpy as np

## 2. Load Ground Truth

Load ground truth labels from an annotated PNG image. The PNG should use distinct colors for each class (background pixels use a separate color).

In [ ]:
from pathlib import Path

# Get project root
project_root = Path.cwd().parent.parent

# Path to ground truth image (labeled PNG with distinct colors per class)
GT_PATH = project_root / "Data" / "processed" / "Lichens_2" / "labeled_lichens_clean.png"

if not GT_PATH.exists():
    print(f"Ground truth file not found: {GT_PATH}")
    print("Creating synthetic ground truth for demonstration...")
    
    # Create synthetic ground truth for demo
    from spectral_select.types import GroundTruth
    
    # Load spatial shape from processed data
    data_path = project_root / "Data" / "processed" / "LichensProcessed" / "data_cutoff_60nm_exposure_max_power_min.pkl"
    from spectral_select import SpectraData
    data = SpectraData.from_pickle(data_path)
    H, W = data.spatial_shape
    
    # Create synthetic labels (3 classes in different regions)
    labels = np.zeros((H, W), dtype=np.int32)
    labels[:H//3, :] = 0  # Class 0: top region
    labels[H//3:2*H//3, :W//2] = 1  # Class 1: middle-left
    labels[H//3:2*H//3, W//2:] = 2  # Class 2: middle-right
    labels[2*H//3:, :] = 0  # Class 0: bottom region
    
    ground_truth = GroundTruth(
        labels=labels,
        class_names=["substrate", "lichen_type_1", "lichen_type_2"],
        color_mapping={0: (0, 255, 0), 1: (255, 0, 0), 2: (0, 0, 255)},
    )
    print(f"Created synthetic ground truth: {labels.shape}")
else:
    # Load from actual annotated PNG
    ground_truth = load_ground_truth_from_png(
        GT_PATH,
        class_colors={
            "substrate": (0, 255, 0),      # Green
            "lichen_1": (255, 0, 0),       # Red  
            "lichen_2": (0, 0, 255),       # Blue
        },
        color_tolerance=50,  # Euclidean distance tolerance for matching
    )

print(f"Ground truth shape: {ground_truth.labels.shape}")
print(f"Classes: {ground_truth.class_names}")
print(f"Color mapping: {ground_truth.color_mapping}")

## 3. Load Clustering Results

Load your clustering predictions. These should be a 2D array matching the ground truth shape, where each pixel has a cluster label.

In [ ]:
# Path to clustering results  
PREDICTIONS_PATH = project_root / "results" / "clustering" / "cluster_labels.npy"

if not PREDICTIONS_PATH.exists():
    print(f"Predictions file not found: {PREDICTIONS_PATH}")
    print("Creating synthetic predictions for demonstration...")
    
    # Create noisy version of ground truth as "predictions"
    # This simulates clustering results with some error
    H, W = ground_truth.labels.shape
    predictions = ground_truth.labels.copy()
    
    # Add some noise to simulate clustering imperfections
    noise_mask = np.random.random((H, W)) < 0.1  # 10% noise
    random_labels = np.random.randint(0, 3, (H, W))
    predictions[noise_mask] = random_labels[noise_mask]
    
    print(f"Created synthetic predictions with ~10% noise")
else:
    predictions = np.load(PREDICTIONS_PATH)

print(f"Predictions shape: {predictions.shape}")
print(f"Unique clusters: {np.unique(predictions)}")

## 4. Run Validation

Fit the Validator to compute all metrics. The validator handles:
- Optimal cluster-to-GT assignment (using Hungarian algorithm)
- Excluding background pixels (-1) from scoring
- Computing comprehensive metrics

In [ ]:
validator = Validator()
validator.fit(predictions, ground_truth)

# Primary metric: Adjusted Rand Index
print(f"ARI Score: {validator.score():.4f}")

## 5. Explore Metrics

Access all computed metrics through the `metrics` property:

In [ ]:
m = validator.metrics

print("=== Clustering Metrics ===")
print(f"Adjusted Rand Index:     {m.adjusted_rand_score:.4f}")
print(f"Normalized Mutual Info:  {m.normalized_mutual_info:.4f}")
print(f"V-Measure:               {m.v_measure:.4f}")
print(f"Homogeneity:             {m.homogeneity:.4f}")
print(f"Completeness:            {m.completeness:.4f}")
print(f"Fowlkes-Mallows:         {m.fowlkes_mallows_score:.4f}")
print(f"Purity:                  {m.purity:.4f}")
print()
print(f"Predicted Clusters:      {m.n_predicted_clusters}")
print(f"Ground Truth Classes:    {m.n_ground_truth_classes}")

## 6. Per-Class Analysis

Examine performance for each ground truth class:

In [ ]:
print("=== Per-Class Metrics ===")
print(f"{'Class':<10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 45)

# Access per-class metrics from separate dictionaries
for class_id in sorted(m.per_class_f1.keys()):
    prec = m.per_class_precision.get(class_id, 0.0)
    rec = m.per_class_recall.get(class_id, 0.0)
    f1 = m.per_class_f1.get(class_id, 0.0)
    print(f"{class_id:<10} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

## 7. Generate Report

Create a comprehensive Markdown-formatted report:

In [ ]:
report = validator.generate_report()
print(report)

Optionally save the report to a file:

In [ ]:
# from pathlib import Path
# validator.generate_report(output_path=Path("validation_report.md"))

## 8. Visualize Results

Create visualizations using `Visualizer.from_validation()`:

In [ ]:
# Create visualizer from validation results
viz = Visualizer.from_validation(validator, predictions)

In [ ]:
# Confusion matrix: rows = true class, columns = predicted cluster
viz.plot_confusion_matrix(
    cm=validator.metrics.confusion_matrix,
    class_names=ground_truth.class_names if ground_truth.class_names else None,
)

In [ ]:
# Per-class metrics bar chart
# Construct the expected format from separate metric dicts
per_class_data = {}
for class_id in m.per_class_f1.keys():
    per_class_data[class_id] = {
        'precision': m.per_class_precision.get(class_id, 0.0),
        'recall': m.per_class_recall.get(class_id, 0.0),
        'f1': m.per_class_f1.get(class_id, 0.0),
        'support': int(np.sum(ground_truth.labels == class_id)),
    }

viz.plot_per_class_metrics(per_class_data)

In [ ]:
# Spatial accuracy heatmap
viz.plot_accuracy_heatmap(
    ground_truth=ground_truth.labels,
    predictions=predictions,
)

## Summary

In this notebook, you learned how to:

1. **Load ground truth** with `load_ground_truth_from_png()`
2. **Run validation** with `Validator.fit(predictions, ground_truth)`
3. **Get metrics** with `validator.score()` and `validator.metrics`
4. **Generate reports** with `validator.generate_report()`
5. **Visualize** with `Visualizer.from_validation()`

**Key metrics explained:**
- **ARI** (Adjusted Rand Index): Measures agreement between clusterings, adjusted for chance. Range: [-1, 1], higher is better.
- **NMI** (Normalized Mutual Info): Information-theoretic measure of clustering quality. Range: [0, 1].
- **Purity**: Fraction of majority class in each cluster. Simple but sensitive to cluster count.
- **V-Measure**: Harmonic mean of homogeneity and completeness.

**Next steps:**
- See `01_quickstart.ipynb` for the full wavelength selection workflow
- Try different clustering algorithms on the selected wavelengths
- Compare validation metrics across different band selections